In [ ]:
from google.colab import files
uploaded = files.upload()

Imports

In [ ]:
!pip install catboost
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, classification_report, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor, CatBoostClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

Dataset Loading and Preprocessing

In [ ]:
df = pd.read_csv("crimes.csv")

df['Incident_Date'] = pd.to_datetime(df['Incident_Date'])
df['Year'] = df['Incident_Date'].dt.year
df['Month'] = df['Incident_Date'].dt.month
df['Day_of_Week'] = df['Incident_Date'].dt.day_name()

Feature Mapping

In [ ]:
crime_map = {
    # THEFT
    'THEFT OF IDENTITY': 'THEFT',
    'THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND OVER)': 'THEFT',
    'THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER)': 'THEFT',
    'THEFT PLAIN - PETTY ($950 & UNDER)': 'THEFT',
    'THEFT-GRAND ($950.01 & OVER)EXCPT,GUNS,FOWL,LIVESTK,PROD': 'THEFT',
    'THEFT, PERSON': 'THEFT',
    'THEFT PLAIN - ATTEMPT': 'THEFT',
    'THEFT FROM PERSON - ATTEMPT': 'THEFT',
    'BUNCO, PETTY THEFT': 'THEFT',
    'BUNCO, ATTEMPT': 'THEFT',
    'BUNCO, GRAND THEFT': 'THEFT',
    'SHOPLIFTING - PETTY THEFT ($950 & UNDER)': 'THEFT',
    'SHOPLIFTING-GRAND THEFT ($950.01 & OVER)': 'THEFT',
    'SHOPLIFTING - ATTEMPT': 'THEFT',
    'TILL TAP - PETTY ($950 & UNDER)': 'THEFT',
    'TILL TAP - GRAND THEFT ($950.01 & OVER)': 'THEFT',
    'PURSE SNATCHING': 'THEFT',
    'PURSE SNATCHING - ATTEMPT': 'THEFT',
    'EMBEZZLEMENT, GRAND THEFT ($950.01 & OVER)': 'THEFT',
    'EMBEZZLEMENT, PETTY THEFT ($950 & UNDER)': 'THEFT',
    'DEFRAUDING INNKEEPER/THEFT OF SERVICES, $950 & UNDER': 'THEFT',
    'DEFRAUDING INNKEEPER/THEFT OF SERVICES, OVER $950.01': 'THEFT',
    'DOCUMENT FORGERY / STOLEN FELONY': 'THEFT',
    'DOCUMENT WORTHLESS ($200 & UNDER)': 'THEFT',
    'DOCUMENT WORTHLESS ($200.01 & OVER)': 'THEFT',
    'CREDIT CARDS, FRAUD USE ($950 & UNDER': 'THEFT',
    'CREDIT CARDS, FRAUD USE ($950.01 & OVER)': 'THEFT',
    'PETTY THEFT - AUTO REPAIR': 'THEFT',

    # BURGLARY
    'BURGLARY': 'BURGLARY',
    'BURGLARY FROM VEHICLE': 'BURGLARY',
    'BURGLARY, ATTEMPTED': 'BURGLARY',
    'BURGLARY FROM VEHICLE, ATTEMPTED': 'BURGLARY',

    # ROBBERY
    'ROBBERY': 'ROBBERY',
    'ATTEMPTED ROBBERY': 'ROBBERY',

    # ASSAULT
    'ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT': 'ASSAULT',
    'INTIMATE PARTNER - SIMPLE ASSAULT': 'ASSAULT',
    'INTIMATE PARTNER - AGGRAVATED ASSAULT': 'ASSAULT',
    'BATTERY - SIMPLE ASSAULT': 'ASSAULT',
    'BATTERY WITH SEXUAL CONTACT': 'ASSAULT',
    'BATTERY POLICE (SIMPLE)': 'ASSAULT',
    'CHILD ABUSE (PHYSICAL) - SIMPLE ASSAULT': 'ASSAULT',
    'CHILD ABUSE (PHYSICAL) - AGGRAVATED ASSAULT': 'ASSAULT',
    'CRIMINAL THREATS - NO WEAPON DISPLAYED': 'ASSAULT',
    'OTHER ASSAULT': 'ASSAULT',
    'ASSAULT WITH DEADLY WEAPON ON POLICE OFFICER': 'ASSAULT',
    'VIOLATION OF COURT ORDER': 'ASSAULT',
    'VIOLATION OF RESTRAINING ORDER': 'ASSAULT',
    'VIOLATION OF TEMPORARY RESTRAINING ORDER': 'ASSAULT',
    'CONTEMPT OF COURT': 'ASSAULT',

    # SEXUAL OFFENSE
    'LEWD/LASCIVIOUS ACTS WITH CHILD': 'SEXUAL OFFENSE',
    'SEX,UNLAWFUL(INC MUTUAL CONSENT, PENETRATION W/ FRGN OBJ': 'SEXUAL OFFENSE',
    'ORAL COPULATION': 'SEXUAL OFFENSE',
    'SEXUAL PENETRATION W/FOREIGN OBJECT': 'SEXUAL OFFENSE',
    'LEWD CONDUCT': 'SEXUAL OFFENSE',
    'RAPE, ATTEMPTED': 'SEXUAL OFFENSE',
    'RAPE, FORCIBLE': 'SEXUAL OFFENSE',
    'CHILD PORNOGRAPHY': 'SEXUAL OFFENSE',
    'PANDERING': 'SEXUAL OFFENSE',
    'PROSTITUTION / PIMPING': 'SEXUAL OFFENSE',

    # VEHICLE
    'VEHICLE - STOLEN': 'VEHICLE',
    'VEHICLE - ATTEMPT STOLEN': 'VEHICLE',
    'VEHICLE, STOLEN - OTHER (MOTORIZED SCOOTERS, BIKES, ETC)': 'VEHICLE',
    'THROWING OBJECT AT MOVING VEHICLE': 'VEHICLE',
    'DRIVING WITHOUT OWNER CONSENT (DWOC)': 'VEHICLE',

    # VANDALISM
    'VANDALISM - MISDEAMEANOR ($399 OR UNDER)': 'VANDALISM',
    'VANDALISM - FELONY ($400 & OVER, ALL CHURCH VANDALISMS)': 'VANDALISM',

    # TRESPASS
    'TRESPASSING': 'TRESPASS',

    # OTHER (everything else)
}

# Apply generalized crime mapping
df['Crime_Type'] = df['Crime_Type'].map(crime_map).fillna('OTHER')



# ---------------------------
# Weapon Map (generalized)
# ---------------------------
weapon_map = {
    # Main categories
    'UNKNOWN': 'UNKNOWN',
    'HAND GUN': 'FIREARM',
    'REVOLVER': 'FIREARM',
    'RIFLE': 'FIREARM',
    'SEMI-AUTOMATIC PISTOL': 'FIREARM',
    'SEMI-AUTOMATIC RIFLE': 'FIREARM',
    'AUTOMATIC WEAPON/SUB-MACHINE GUN': 'FIREARM',
    'ASSAULT WEAPON/UZI/AK47/ETC': 'FIREARM',
    'UZI SEMIAUTOMATIC ASSAULT RIFLE': 'FIREARM',
    'MAC-10 SEMIAUTOMATIC ASSAULT WEAPON': 'FIREARM',
    'MAC-11 SEMIAUTOMATIC ASSAULT WEAPON': 'FIREARM',
    'M-14 SEMIAUTOMATIC ASSAULT RIFLE': 'FIREARM',
    'M1-1 SEMIAUTOMATIC ASSAULT RIFLE': 'FIREARM',
    'HECKLER & KOCH 91 SEMIAUTOMATIC ASSAULT RIFLE': 'FIREARM',
    'HECKLER & KOCH 93 SEMIAUTOMATIC ASSAULT RIFLE': 'FIREARM',
    'SAWED OFF RIFLE/SHOTGUN': 'FIREARM',
    'SHOTGUN': 'FIREARM',
    'ANTIQUE FIREARM': 'FIREARM',
    'RELIC FIREARM': 'FIREARM',
    'STARTER PISTOL/REVOLVER': 'FIREARM',
    'TOY GUN': 'FIREARM',
    'SIMULATED GUN': 'FIREARM',
    'OTHER FIREARM': 'FIREARM',

    'KNIFE WITH BLADE 6INCHES OR LESS': 'KNIFE',
    'KNIFE WITH BLADE OVER 6 INCHES IN LENGTH': 'KNIFE',
    'FOLDING KNIFE': 'KNIFE',
    'OTHER KNIFE': 'KNIFE',
    'BOWIE KNIFE': 'KNIFE',
    'KITCHEN KNIFE': 'KNIFE',
    'MACHETE': 'KNIFE',
    'STRAIGHT RAZOR': 'KNIFE',
    'RAZOR': 'KNIFE',
    'RAZOR BLADE': 'KNIFE',
    'ICE PICK': 'KNIFE',
    'DIRK/DAGGER': 'KNIFE',
    'SCISSORS': 'KNIFE',
    'CLEAVER': 'KNIFE',
    'SWORD': 'KNIFE',
    'SWITCH BLADE': 'KNIFE',
    'SYRINGE': 'KNIFE',

    'STICK': 'BLUNT',
    'PIPE/METAL PIPE': 'BLUNT',
    'CLUB/BAT': 'BLUNT',
    'HAMMER': 'BLUNT',
    'TIRE IRON': 'BLUNT',
    'BOARD': 'BLUNT',
    'ROCK/THROWN OBJECT': 'BLUNT',
    'CONCRETE BLOCK/BRICK': 'BLUNT',
    'BOTTLE': 'BLUNT',
    'BRASS KNUCKLES': 'BLUNT',
    'BLUNT INSTRUMENT': 'BLUNT',
    'GLASS': 'BLUNT',

    'MACE/PEPPER SPRAY': 'NON-LETHAL',
    'STUN GUN': 'NON-LETHAL',
    'BOW AND ARROW': 'NON-LETHAL',
    'PHYSICAL PRESENCE': 'NON-LETHAL',

    'FIRE': 'ORGANIC/HAZARD',
    'SCALDING LIQUID': 'ORGANIC/HAZARD',
    'CAUSTIC CHEMICAL/POISON': 'ORGANIC/HAZARD',
    'BOMB THREAT': 'ORGANIC/HAZARD',
    'EXPLOXIVE DEVICE': 'ORGANIC/HAZARD',
    'DOG/ANIMAL (SIC ANIMAL ON)': 'ORGANIC/HAZARD',

    'VEHICLE': 'VEHICLE',
    'VERBAL THREAT': 'THREAT',
    'DEMAND NOTE': 'THREAT',
}

df['Weapon_Cleaned'] = df['Weapon_Cleaned'].map(weapon_map).fillna('OTHER')


# ---------------------------
# Premise Map (generalized)
# ---------------------------
premise_map = {
    'SINGLE FAMILY DWELLING': 'RESIDENCE',
    'MULTI-UNIT DWELLING (APARTMENT, DUPLEX, ETC)': 'RESIDENCE',
    'PORCH, RESIDENTIAL': 'RESIDENCE',
    'OTHER RESIDENCE': 'RESIDENCE',
    'GARAGE/CARPORT': 'RESIDENCE',
    'CONDOMINIUM/TOWNHOUSE': 'RESIDENCE',
    'SHORT-TERM VACATION RENTAL': 'RESIDENCE',
    'TRANSITIONAL HOUSING/HALFWAY HOUSE': 'RESIDENCE',
    'GROUP HOME': 'RESIDENCE',
    "SINGLE RESIDENCE OCCUPANCY (SRO'S) LOCATIONS": 'RESIDENCE',

    'SIDEWALK': 'STREET/OUTDOORS',
    'STREET': 'STREET/OUTDOORS',
    'ALLEY': 'STREET/OUTDOORS',
    'YARD (RESIDENTIAL/BUSINESS)': 'STREET/OUTDOORS',
    'OTHER/OUTSIDE': 'STREET/OUTDOORS',
    'VACANT LOT': 'STREET/OUTDOORS',
    'UNDERPASS/BRIDGE*': 'STREET/OUTDOORS',
    'PARK/PLAYGROUND': 'STREET/OUTDOORS',
    'BEACH': 'STREET/OUTDOORS',
    'PATIO*': 'STREET/OUTDOORS',
    'DRIVEWAY': 'STREET/OUTDOORS',

    'RESTAURANT/FAST FOOD': 'BUSINESS',
    'BAR/SPORTS BAR (OPEN DAY & NIGHT)': 'BUSINESS',
    'BAR/COCKTAIL/NIGHTCLUB': 'BUSINESS',
    'MARKET': 'BUSINESS',
    'DEPARTMENT STORE': 'BUSINESS',
    'MINI-MART': 'BUSINESS',
    'LIQUOR STORE': 'BUSINESS',
    'CELL PHONE STORE': 'BUSINESS',
    'CLOTHING STORE': 'BUSINESS',
    'ELECTRONICS STORE (IE:RADIO SHACK, ETC.)': 'BUSINESS',
    'BEAUTY SUPPLY STORE': 'BUSINESS',
    'NAIL SALON': 'BUSINESS',
    'BEAUTY/BARBER SHOP': 'BUSINESS',
    'PET STORE': 'BUSINESS',
    'BOOK STORE': 'BUSINESS',
    'JEWELRY STORE': 'BUSINESS',
    'HARDWARE/BUILDING SUPPLY': 'BUSINESS',
    'AUTO DEALERSHIP (CHEVY, FORD, BMW, MERCEDES, ETC.)': 'BUSINESS',
    'AUTO REPAIR SHOP': 'BUSINESS',
    'CAR WASH': 'BUSINESS',
    'PAWN SHOP': 'BUSINESS',
    'OTHER BUSINESS': 'BUSINESS',
    'SHOPPING MALL (COMMON AREA)': 'BUSINESS',
    'THE BEVERLY CENTER': 'BUSINESS',
    'THE BEVERLY CONNECTION': 'BUSINESS',
    'THE GROVE': 'BUSINESS',
    'BUSINESS OFFICE/OFFICE BUILDING': 'BUSINESS',
    'FACTORY': 'BUSINESS',
    'WAREHOUSE': 'BUSINESS',
    'CONSTRUCTION SITE': 'BUSINESS',
    'PUBLIC STORAGE': 'BUSINESS',
    'STORAGE SHED': 'BUSINESS',
    'TOOL SHED*': 'BUSINESS',
    'HOTEL': 'BUSINESS',
    'MOTEL': 'BUSINESS',

    'TRANSPORTATION FACILITY (AIRPORT)': 'TRANSPORT',
    'BUS STOP': 'TRANSPORT',
    'TRAIN DEPOT/TERMINAL, OTHER THAN MTA': 'TRANSPORT',
    'METROLINK TRAIN': 'TRANSPORT',
    'AMTRAK TRAIN': 'TRANSPORT',
    'GREYHOUND OR INTERSTATE BUS': 'TRANSPORT',
    'MTA BUS': 'TRANSPORT',
    'MTA PROPERTY OR PARKING LOT': 'TRANSPORT',
    'AIRCRAFT': 'TRANSPORT',
    'TAXI': 'TRANSPORT',

    'VEHICLE, PASSENGER/TRUCK': 'VEHICLE',
    'TRUCK, COMMERICAL': 'VEHICLE',
    'VEHICLE STORAGE LOT (CARS, TRUCKS, RV\'S, BOATS, TRAILERS, ETC.)': 'VEHICLE',

    'HIGH SCHOOL': 'SCHOOL',
    'JUNIOR HIGH SCHOOL': 'SCHOOL',
    'ELEMENTARY SCHOOL': 'SCHOOL',
    'PRIVATE SCHOOL/PRESCHOOL': 'SCHOOL',
    'COLLEGE/JUNIOR COLLEGE/UNIVERSITY': 'SCHOOL',
    'SPECIALTY SCHOOL/OTHER': 'SCHOOL',

    'HOSPITAL': 'HOSPITAL',
    'NURSING/CONVALESCENT/RETIREMENT HOME': 'HOSPITAL',
    'HOSPICE': 'HOSPITAL',
    'MEDICAL/DENTAL OFFICES': 'HOSPITAL',
    'MEDICAL MARIJUANA FACILITIES/BUSINESSES': 'HOSPITAL',

}

df['Premise_Type'] = df['Premise_Type'].map(premise_map).fillna('OTHER')

print(df['Weapon_Cleaned'].unique())
print(df['Crime_Type'].unique())
print(df['Premise_Type'].unique())

Feature Extraction

In [ ]:
severe_types = ['ROBBERY', 'ASSAULT', 'SEXUAL OFFENSE', 'ARSON', 'BURGLARY', 'HUMAN TRAFFICKING']
df['Severity'] = df['Crime_Type'].apply(lambda x: 'Severe' if x in severe_types else 'Non-Severe')

crime_monthly = (
    df.groupby(['Area_Name', pd.Grouper(key='Incident_Date', freq='ME')])
    .size()
    .reset_index(name='crime_count')
)

crime_daily = (
    df
    .groupby(['Area_Name', 'Incident_Date'])
    .size()
    .reset_index(name='crime_count')
)
df.to_csv("cleaned.csv", index=False)
print("preprocess done")

# **Predicting Crime Rates 06-25 to 06-27**

## **Correlation**

In [ ]:
cols_for_corr = ['Incident_Date', 'Year', 'Month', 'Day of Week', 'Area_Name', 'Crime_Type',
                   'Weapon_Cleaned', 'Premise_Type', 'Victim_Gender', 'Victim_Age_Group']
df_corr = df[cols_for_corr].copy()

for col in df_corr.columns:
    if df_corr[col].dtype == 'object' or df_corr[col].dtype.name == 'category':
        df_corr[col] = df_corr[col].astype('category').cat.codes

corr_matrix = df_corr.corr()

plt.figure(figsize=(8,8))
sns.heatmap(corr_matrix, cmap='coolwarm', annot=True, square=True, fmt='.2f')
plt.title("Correlation Heatmap of Relevant Variables", fontsize=18)
plt.show()


# **CB**

In [ ]:
daily = df.groupby(['Area_Name','Crime_Type','Incident_Date']).size().reset_index(name='count')
daily = daily.sort_values(['Area_Name','Crime_Type','Incident_Date'])

daily['Incident_Date'] = pd.to_datetime(daily['Incident_Date'])
daily['Year'] = daily['Incident_Date'].dt.year
daily['Month'] = daily['Incident_Date'].dt.month
daily['Week'] = daily['Incident_Date'].dt.isocalendar().week.astype(int)
daily['Day'] = daily['Incident_Date'].dt.day

daily['lag_1'] = daily.groupby(['Area_Name','Crime_Type'])['count'].shift(1)
daily['lag_2'] = daily.groupby(['Area_Name','Crime_Type'])['count'].shift(2)
daily = daily.dropna()

severe_types = ['ROBBERY', 'ASSAULT', 'SEXUAL OFFENSE', 'ARSON', 'BURGLARY', 'HUMAN TRAFFICKING']
daily['Severity'] = df['Crime_Type'].apply(lambda x: 'Severe' if x in severe_types else 'Non-Severe')

X = daily[['Area_Name','Crime_Type','Year','Month','Week','Day','lag_1','lag_2', 'Severity']]
y = daily['count']

model = CatBoostRegressor(iterations=5000, learning_rate=0.9, verbose=True)
model.fit(X, y, cat_features=[0,1,8])

preds = model.predict(X)
eval_r2 = r2_score(y, preds)
print(f"R^2 score {eval_r2}")

areas = daily['Area_Name'].unique()
types = daily['Crime_Type'].unique()
last_date = daily['Incident_Date'].max()
future_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=730)

future_list = []
for a in areas:
    for t in types:
        z = daily[(daily['Area_Name']==a) & (daily['Crime_Type']==t)]
        if len(z) < 2:
            continue
        l1 = z['count'].iloc[-1]
        l2 = z['count'].iloc[-2]
        f = pd.DataFrame({
            'Incident_Date': future_dates,
            'Area_Name': a,
            'Crime_Type': t,
            'Severity': 'Severe' if t in severe_types else 'Non-Severe',
            'Year': future_dates.year,
            'Month': future_dates.month,
            'Week': future_dates.isocalendar().week.astype(int),
            'Day': future_dates.day,
            'lag_1': l1,
            'lag_2': l2
        })
        future_list.append(f)

future = pd.concat(future_list, ignore_index=True)

X_future = future[['Area_Name','Crime_Type','Year','Month','Week','Day','lag_1','lag_2', 'Severity']]
future['Predicted_Count'] = model.predict(X_future).round().astype(int)

future.to_csv("crime_type_forecast.csv", index=False)
print(f"saved. Total crimes predicted {future['Predicted_Count'].sum()}")

## **Predictions Dataset**

In [ ]:
df['Incident_Date'] = pd.to_datetime(df['Incident_Date'])
df['Month'] = df['Incident_Date'].dt.month
df['Year'] = df['Incident_Date'].dt.year
df['Week'] = df['Incident_Date'].dt.isocalendar().week.astype(int)
df['Day'] = df['Incident_Date'].dt.day

base_features = ['Area_Name', 'Crime_Type', 'Year', 'Month', 'Week', 'Day', 'Severity']

print("Training")

X = df[base_features]
y = df['Weapon_Cleaned']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

weapon_model = CatBoostClassifier(iterations=10, learning_rate=1, verbose=True, random_state =42, loss_function='MultiClass')
weapon_model.fit(X_train, y_train, cat_features=[0,1,6])

y_pred = weapon_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="macro", zero_division=0)
recall = recall_score(y_test, y_pred, average="macro", zero_division=0)
f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

print(f"\nAccuracy: {acc:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1: {f1:.3f}")

report = classification_report(y_test, y_pred, output_dict=True)
report_df = pd.DataFrame(report).iloc[:-1, :].T
plt.figure(figsize=(10, 6))
sns.heatmap(report_df, annot=True, cmap="Blues", fmt=".2f")
plt.title("Classification Report Heatmap")
plt.ylabel("Metrics")
plt.xlabel("Classes")
plt.show()

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix Heatmap")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
features = ['Area_Name', 'Crime_Type', 'Year', 'Month', 'Week', 'Day', 'Severity', 'Weapon_Cleaned']
print("Training")

X = df[features]
y = df['Victim_Gender']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

age_model = CatBoostClassifier(iterations=500, learning_rate=0.05, verbose=True)
age_model.fit(X_train, y_train, cat_features=[0,1,6])

y_pred = age_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="macro", zero_division=0)
recall = recall_score(y_test, y_pred, average="macro", zero_division=0)
f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

print(f"\nAccuracy: {acc:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1: {f1:.3f}")

report = classification_report(y_test, y_pred, output_dict=True)
report_df = pd.DataFrame(report)
plt.figure(figsize=(10, 6))
sns.heatmap(report_df, annot=True, cmap="Blues", fmt=".2f")
plt.title("Classification Report Heatmap")
plt.ylabel("Metrics")
plt.xlabel("Classes")
plt.show()

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix Heatmap")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
features = ['Area_Name', 'Crime_Type', 'Year', 'Month', 'Week', 'Day', 'Severity', 'Weapon_Cleaned', 'Victim_Gender']
print("Training")

X = df[features]
y = df['Victim_Age_Group']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

gen_model = CatBoostClassifier(iterations=500, learning_rate=0.05, verbose=True)
gen_model.fit(X_train, y_train, cat_features=[0,1,6])

y_pred = gen_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="macro", zero_division=0)
recall = recall_score(y_test, y_pred, average="macro", zero_division=0)
f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

print(f"\nAccuracy: {acc:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1: {f1:.3f}")

report = classification_report(y_test, y_pred, output_dict=True)
report_df = pd.DataFrame(report)
plt.figure(figsize=(10, 6))
sns.heatmap(report_df, annot=True, cmap="Blues", fmt=".2f")
plt.title("Classification Report Heatmap")
plt.ylabel("Metrics")
plt.xlabel("Classes")
plt.show()

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix Heatmap")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
features = ['Area_Name', 'Crime_Type', 'Year', 'Month', 'Week', 'Day', 'Severity', 'Weapon_Cleaned', 'Victim_Gender', 'Victim_Age_Group']
print("Training")

X = df[features]
y = df['Premise_Type']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pre_model = CatBoostClassifier(iterations=500, learning_rate=0.05, verbose=True)
pre_model.fit(X_train, y_train, cat_features=[0,1])

y_pred = pre_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="macro", zero_division=0)
recall = recall_score(y_test, y_pred, average="macro", zero_division=0)
f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

print(f"\nAccuracy: {acc:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1: {f1:.3f}")

report = classification_report(y_test, y_pred, output_dict=True)
report_df = pd.DataFrame(report)
plt.figure(figsize=(10, 6))
sns.heatmap(report_df, annot=True, cmap="Blues", fmt=".2f")
plt.title("Classification Report Heatmap")
plt.ylabel("Metrics")
plt.xlabel("Classes")
plt.show()

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix Heatmap")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()